# Notebook 3 — Stage 1: Label Data with GPT (Teacher)
**MAI 656 — Natural Language Processing | Canadian University Dubai | Spring 2026**

This notebook uses GPT-4o as a **teacher model** to generate transcription labels for Arabic handwriting images.

> 💡 **This stage does NOT require a GPU.** Run it on your local machine, Colab (free tier), or any machine with internet access.

**Input:** Batch folders produced by Notebook 2 → `data/raw_png_batches/batch_NNN/`  
**Output:** `data/gpt_output/batch_NNN.jsonl` — one JSONL file per batch folder

**Cost estimate:** ~200 images × $0.01–0.03 each = **$2–6 total**

Set `BATCH_TO_PROCESS` in the configuration cell to run a **single batch** (e.g. `"batch_001"`) or `"all"` to process every batch in one go.

## 1. Install Dependencies

In [1]:
!pip install openai python-dotenv --quiet

## 2. Set Project Root

**Colab:** mounts Google Drive and reads from `MyDrive/nlp_project`.  
**Local:** update `LOCAL_PROJECT_ROOT` below to point to your project folder.

In [2]:
import os
from pathlib import Path

IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/nlp_project')
else:
    # ── Set this to your local project directory ──────────────────────────
    PROJECT_ROOT = Path(r'C:\Users\mitah\github_projects\nlp_project')
    # ──────────────────────────────────────────────────────────────────────

assert PROJECT_ROOT.exists(), (
    f'Project root not found: {PROJECT_ROOT}\n'
    'Update LOCAL_PROJECT_ROOT above to match your system.'
)

print(f'{"Colab" if IN_COLAB else "Local"} | Project root: {PROJECT_ROOT}')

Local | Project root: C:\Users\mitah\github_projects\nlp_project


## 3. Set Your OpenAI API Key

**Option A — Enter directly (Colab):** Fill in the cell below.  
**Option B — `.env` file (local):** Create a `.env` file with `OPENAI_API_KEY=sk-...` and `load_dotenv()` will pick it up automatically.

In [3]:
import os
from dotenv import load_dotenv

# Load from .env file if it exists (local usage)
load_dotenv()

# If OPENAI_API_KEY is not set yet, set it here (Colab usage)
if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = 'sk-your-key-here'  # <-- replace

# Verify the key is set
key = os.environ.get('OPENAI_API_KEY', '')
if not key or key == 'sk-your-key-here':
    raise ValueError('Please set your OPENAI_API_KEY before continuing.')
print(f'API key loaded: {key[:8]}...')

API key loaded: sk-proj-...


## 4. Configure Paths

In [4]:
from pathlib import Path

BATCH_DIR  = Path(PROJECT_ROOT) / 'sample_training_data'
OUTPUT_DIR = Path(PROJECT_ROOT) / 'sample_training_data' / 'gpt_output'

# Set to a specific folder name (e.g. "batch_001") to process one batch,
# or "all" to process every batch folder under data/raw_png_batches/.
BATCH_TO_PROCESS = "raw_png"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Resolve the list of (batch_name, images, output_file) jobs to run
if BATCH_TO_PROCESS == "all":
    batch_dirs = sorted(BATCH_DIR.glob('batch_*'))
    if not batch_dirs:
        print(f'WARNING: No batch folders found under {BATCH_DIR}')
        print('Run Notebook 2 (data preprocessing) first.')
else:
    target = BATCH_DIR / BATCH_TO_PROCESS
    if not target.exists():
        raise FileNotFoundError(f'Batch folder not found: {target}')
    batch_dirs = [target]

jobs = []
for d in batch_dirs:
    imgs = sorted(d.glob('*.png')) + sorted(d.glob('*.jpg'))
    jobs.append((d.name, imgs, OUTPUT_DIR / f'{d.name}.jsonl'))

total_images = sum(len(imgs) for _, imgs, _ in jobs)
print(f'Batch selection: {BATCH_TO_PROCESS}')
print(f'Batches to process: {len(jobs)}')
print(f'Images to label: {total_images}')

Batch selection: raw_png
Batches to process: 1
Images to label: 100


## 5. Define the Labeling Function

In [5]:
import openai
import base64
import json

client = openai.OpenAI()

SYSTEM_PROMPT = """You are an Arabic handwriting recognition system.
Read the handwritten Arabic text in this image and transcribe it exactly
as written. Return your response as JSON:
{
    "transcription": "the Arabic text you read",
    "confidence": "high/medium/low"
}
Return ONLY the JSON, no other text."""

def label_image(image_path: str) -> dict:
    """Send an image to GPT and get the transcription."""
    with open(image_path, 'rb') as f:
        image_data = base64.b64encode(f.read()).decode('utf-8')

    ext = Path(image_path).suffix.lower()
    mime = 'image/png' if ext == '.png' else 'image/jpeg'

    response = client.chat.completions.create(
        model='gpt-5.4',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': [
                {'type': 'image_url', 'image_url': {
                    'url': f'data:{mime};base64,{image_data}'
                }}
            ]}
        ],
        max_completion_tokens=2048,
        temperature=0.0
    )

    return {
        'image_path': str(image_path),
        'gpt_response': response.choices[0].message.content,
        'model': response.model,
        'usage': {
            'prompt_tokens': response.usage.prompt_tokens,
            'completion_tokens': response.usage.completion_tokens
        }
    }

print('label_image() defined.')

label_image() defined.


## 6. Run the Labeling Pipeline

Each batch writes to its own `data/gpt_output/batch_NNN.jsonl` file. Results are appended so you can safely resume if interrupted.

In [6]:
errors = 0

for batch_name, batch_images, output_file in jobs:
    print(f'\n=== {batch_name} → {output_file.name} ===')

    # Load already-processed paths so we can resume safely
    processed = set()
    if output_file.exists():
        with open(output_file, encoding='utf-8') as f:
            for line in f:
                try:
                    record = json.loads(line)
                    processed.add(record['image_path'])
                except Exception:
                    pass
        print(f'Resuming: {len(processed)} images already labeled.')

    remaining = [img for img in batch_images if str(img) not in processed]
    print(f'{len(remaining)} images left to label.')

    with open(output_file, 'a', encoding='utf-8') as f:
        for i, img_path in enumerate(remaining):
            print(f'[{i+1}/{len(remaining)}] Labeling {img_path.name}...', end=' ')
            try:
                result = label_image(str(img_path))
                f.write(json.dumps(result, ensure_ascii=False) + '\n')
                f.flush()
                print('OK')
            except Exception as e:
                print(f'ERROR: {e}')
                errors += 1
                continue

print(f'\nDone! Labels saved under {OUTPUT_DIR}')
print(f'Errors: {errors}')


=== raw_png → raw_png.jsonl ===
100 images left to label.
[1/100] Labeling AHTD3A0001_Para2_3.png... OK
[2/100] Labeling AHTD3A0001_Para2_4.png... OK
[3/100] Labeling AHTD3A0001_Para3_1.png... OK
[4/100] Labeling AHTD3A0001_Para3_2.png... OK
[5/100] Labeling AHTD3A0002_Para2_1.png... OK
[6/100] Labeling AHTD3A0002_Para2_2.png... OK
[7/100] Labeling AHTD3A0002_Para2_3.png... OK
[8/100] Labeling AHTD3A0002_Para2_4.png... OK
[9/100] Labeling AHTD3A0002_Para3_1.png... OK
[10/100] Labeling AHTD3A0002_Para3_2.png... OK
[11/100] Labeling AHTD3A0002_Para3_3.png... OK
[12/100] Labeling AHTD3A0002_Para3_4.png... OK
[13/100] Labeling AHTD3A0002_Para3_5.png... OK
[14/100] Labeling AHTD3A0002_Para3_6.png... OK
[15/100] Labeling AHTD3A0004_Para2_1.png... OK
[16/100] Labeling AHTD3A0004_Para2_2.png... OK
[17/100] Labeling AHTD3A0004_Para2_4.png... OK
[18/100] Labeling AHTD3A0004_Para2_5.png... OK
[19/100] Labeling AHTD3A0004_Para3_1.png... OK
[20/100] Labeling AHTD3A0004_Para3_2.png... OK
[21/100] L

## 7. Inspect the Output

In [8]:
# Show the first 3 records across all batch output files
all_lines = []
for jsonl_file in sorted(OUTPUT_DIR.glob('raw_png*.jsonl')):
    with open(jsonl_file, encoding='utf-8') as f:
        all_lines.extend(f.readlines())

print(f'Total labeled samples: {len(all_lines)}\n')
for i, line in enumerate(all_lines[:3]):
    record = json.loads(line)
    print(f'--- Sample {i+1} ---')
    print(f"  Image:    {record['image_path']}")
    print(f"  Response: {record['gpt_response']}")
    print(f"  Tokens:   {record['usage']}")
    print()

Total labeled samples: 100

--- Sample 1 ---
  Image:    C:\Users\mitah\github_projects\nlp_project\sample_training_data\raw_png\AHTD3A0001_Para2_3.png
  Response: {"transcription":"من التجارب في الأسرة عاطفة فضل من رزقهم الله تعالى وهي به كسب بالعقل الذين هدروا ما","confidence":"medium"}
  Tokens:   {'prompt_tokens': 374, 'completion_tokens': 38}

--- Sample 2 ---
  Image:    C:\Users\mitah\github_projects\nlp_project\sample_training_data\raw_png\AHTD3A0001_Para2_4.png
  Response: {"transcription":"لجميع الإخوة والذين يقدروا أحد في الدنيا كالهواء معيشته.","confidence":"medium"}
  Tokens:   {'prompt_tokens': 1891, 'completion_tokens': 32}

--- Sample 3 ---
  Image:    C:\Users\mitah\github_projects\nlp_project\sample_training_data\raw_png\AHTD3A0001_Para3_1.png
  Response: {"transcription":"فقال للمؤمنين في منازل آبائك وأجدادك من المجابرة النيل أسو الكدر","confidence":"low"}
  Tokens:   {'prompt_tokens': 461, 'completion_tokens': 37}



## 8. Usage & Cost Summary

In [10]:
total_prompt = 0
total_completion = 0

for jsonl_file in sorted(OUTPUT_DIR.glob('raw_png*.jsonl')):
    with open(jsonl_file, encoding='utf-8') as f:
        for line in f:
            r = json.loads(line)
            total_prompt += r['usage']['prompt_tokens']
            total_completion += r['usage']['completion_tokens']

# GPT-4o pricing as of April 2026: ~$2.50/1M input, ~$15/1M output
input_cost  = total_prompt     / 1_000_000 * 2.50
output_cost = total_completion / 1_000_000 * 15.00
total_cost  = input_cost + output_cost

print(f'Total prompt tokens:     {total_prompt:,}')
print(f'Total completion tokens: {total_completion:,}')
print(f'Estimated cost:          ${total_cost:.2f}')

Total prompt tokens:     57,741
Total completion tokens: 3,207
Estimated cost:          $0.19


## Next Step

Labels are saved to `data/gpt_output/batch_NNN.jsonl` (one file per batch). Continue to:

**Notebook 4 → `04_data_transformation.ipynb`** — Convert GPT labels to Qwen training format